# Phase 3: Exploratory Data Analysis (EDA)
## Enhanced Fake News Detection using Machine Learning & NLP

**Course / Academic Project:** Second-Year B.Tech Computer Science & Engineering  
**Dataset:** ISOT Fake & Real News Dataset (`Fake.csv` & `True.csv`)  
**Objective:** Understand data distributions, stylistic cues, vocabulary patterns, and correlations between linguistic markers and fake/real news labels.

### 1. Import Essential Libraries
We use `pandas`, `numpy`, `matplotlib`, `seaborn`, `wordcloud`, and custom project modules from `src`.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import NewsDataLoader
from src.preprocessing import TextPreprocessor
from src.feature_engineering import HandcraftedLinguisticExtractor
from src import utils

print('Libraries successfully imported!')

### 2. Ingest and Validate Dataset
Load the processed dataset generated by Phase 1 Data Engineering pipeline.

In [2]:
data_path = os.path.join(PROJECT_ROOT, 'data', 'processed_news.csv')
df = pd.read_csv(data_path)
print(f'Loaded {len(df):,} news articles with columns: {list(df.columns)}')
df.head(3)

### 3. Class Distribution Analysis
Check whether the binary classes (`Fake = 0`, `Real = 1`) are balanced.

In [3]:
utils.plot_class_distribution(df)
counts = df['label'].value_counts()
print(f'Fake News (0): {counts[0]:,} ({counts[0]/len(df):.2%})')
print(f'Real News (1): {counts[1]:,} ({counts[1]/len(df):.2%})')

### 4. Subject and Category Distribution
Examine the distribution of topics (e.g., politicsNews, worldnews, Government News).

In [4]:
utils.plot_subject_distribution(df)
print(df.groupby(['subject', 'label']).size().unstack(fill_value=0))

### 5. Article Length and Word Count Distributions
Check if fake news articles differ systematically in length or word count from real journalism.

In [5]:
utils.plot_length_distribution(df)
df['word_count'] = df['text'].fillna('').apply(lambda x: len(x.split()))
print(df.groupby('label')['word_count'].describe())

### 6. Handcrafted Stylometric Feature Extraction & Correlation
Extract 5 linguistic features (capitalization, punctuation density, digit ratio, title/article lengths) and compute Pearson correlation with the target class.

In [6]:
utils.plot_correlation_heatmap(df)
extractor = HandcraftedLinguisticExtractor()
features = extractor.transform(df.sample(5000, random_state=42))
corr_df = pd.DataFrame(features, columns=extractor.feature_names)
corr_df.describe()

### 7. Lexical Frequency & Top Words
Inspect the top 20 most frequent terms across Fake vs Real news.

In [7]:
utils.plot_top_words(df, top_n=20)

### 8. Word Clouds for Visual Lexical Inspection
Generate word clouds illustrating distinctive vocabulary in each class.

In [8]:
utils.plot_wordclouds(df)
print('Word clouds saved to reports/figures!')

### 9. Key Findings & Conclusion
1. **Vocabulary Signatures:** Real news consistently utilizes formal neutral reporting verbs (`say`, `reuters`, `state`, `official`), whereas fake news contains sensationalist and emotional words (`video`, `shocking`, `watch`, `trump`).
2. **Punctuation & Capitalization:** Fake news headlines exhibit significantly higher frequencies of exclamation marks, question marks, and fully-capitalized words.
3. **Hybrid Feature Justification:** Combining TF-IDF n-grams with handcrafted stylometric features provides the downstream classifiers with both semantic content and stylistic cues!